# ORCA — Oil Spill Detection Experiments

## Objective

Train a semantic segmentation model to detect oil-spill regions in Sentinel-1 SAR imagery.

### Pipeline

Sentinel-1 SAR Image
        ↓
Preprocessing
        ↓
Oil Annotation Parsing
        ↓
Pseudo-Segmentation Mask Generation
        ↓
U-Net
        ↓
Oil Probability Map
        ↓
Binary Spill Mask

## Dataset

PANGAEA Sentinel-1 SAR oil-spill dataset.

The dataset contains:
- Sentinel-1 SAR image patches
- XML object annotations
- Oil-spill bounding boxes
- No-oil image patches

### Important limitation

The available annotations are bounding boxes rather than pixel-level segmentation masks.

Therefore, the initial U-Net baseline converts each oil bounding box into a binary rectangular pseudo-mask.

This model should therefore be treated as a baseline for the SIH prototype. 
Future training should use true pixel-level masks if available.

In [ ]:
import os
import random
import json
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# =========================
# ORCA Detection Configuration
# =========================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 256
BATCH_SIZE = 16
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3

MODEL_DIR = Path("../models/spill_detection")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "unet_best.pt"

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Epochs:", NUM_EPOCHS)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

print("Project root:")
print(PROJECT_ROOT)

print("\nContents:")
for item in PROJECT_ROOT.iterdir():
    print(item)

In [ ]:
image_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}
xml_files = []
image_files = []

for path in SATELLITE_DIR.rglob("*"):
    if path.is_file():
        if path.suffix.lower() in image_extensions:
            image_files.append(path)
        elif path.suffix.lower() == ".xml":
            xml_files.append(path)

print("Images:", len(image_files))
print("XML annotations:", len(xml_files))

print("\nExample images:")
for p in image_files[:10]:
    print(p)

print("\nExample XML files:")
for p in xml_files[:10]:
    print(p)

In [ ]:
xml_path = xml_files[0]

print(xml_path)

tree = ET.parse(xml_path)
root = tree.getroot()

print(ET.tostring(root, encoding="unicode")[:5000])

In [ ]:
def parse_xml_annotation(xml_path):
    """
    Parse oil-spill bounding boxes from an XML annotation.

    Returns:
        list of dictionaries containing:
        xmin, ymin, xmax, ymax
    """

    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes = []

    for obj in root.findall(".//object"):
        bbox = obj.find(".//bndbox")

        if bbox is None:
            continue

        try:
            xmin = float(bbox.findtext("xmin"))
            ymin = float(bbox.findtext("ymin"))
            xmax = float(bbox.findtext("xmax"))
            ymax = float(bbox.findtext("ymax"))
        except (TypeError, ValueError):
            continue

        if xmax <= xmin or ymax <= ymin:
            continue

        boxes.append({
            "xmin": xmin,
            "ymin": ymin,
            "xmax": xmax,
            "ymax": ymax,
        })

    return boxes

In [ ]:
print("IMAGE:", image_files[0].stem)
print("XML:", xml_files[0].stem)
xml_by_stem = {
    p.stem: p
    for p in xml_files
}

records = []

for image_path in image_files:
    annotation_path = xml_by_stem.get(image_path.stem)

    records.append({
        "image": image_path,
        "annotation": annotation_path,
        "has_annotation": annotation_path is not None
    })

df = pd.DataFrame(records)

print(df.head())
print()
print(df["has_annotation"].value_counts())

In [ ]:
total_objects = 0
images_with_oil = 0
images_without_oil = 0

for _, row in df.iterrows():

    annotation = row["annotation"]

    if annotation is None:
        images_without_oil += 1
        continue

    boxes = parse_xml_annotation(annotation)

    if len(boxes) > 0:
        images_with_oil += 1
        total_objects += len(boxes)
    else:
        images_without_oil += 1

print("Total images:", len(df))
print("Images with oil:", images_with_oil)
print("Images without oil:", images_without_oil)
print("Total oil objects:", total_objects)

In [ ]:
def create_pseudo_mask(image_size, boxes):
    """
    Create a binary pseudo-segmentation mask from bounding boxes.

    image_size:
        (width, height)

    boxes:
        list of dictionaries containing bbox coordinates.
    """

    width, height = image_size

    mask = np.zeros((height, width), dtype=np.uint8)

    for box in boxes:

        xmin = max(0, int(round(box["xmin"])))
        ymin = max(0, int(round(box["ymin"])))
        xmax = min(width, int(round(box["xmax"])))
        ymax = min(height, int(round(box["ymax"])))

        if xmax <= xmin or ymax <= ymin:
            continue

        mask[ymin:ymax, xmin:xmax] = 1

    return mask

In [ ]:
sample_row = df[df["has_annotation"]].iloc[0]

image = Image.open(sample_row["image"]).convert("RGB")
boxes = parse_xml_annotation(sample_row["annotation"])

mask = create_pseudo_mask(image.size, boxes)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(image)
axes[0].set_title("Original SAR Image")
axes[0].axis("off")

axes[1].imshow(mask, cmap="gray")
axes[1].set_title("Pseudo Oil Mask")
axes[1].axis("off")

axes[2].imshow(image)
axes[2].imshow(mask, alpha=0.4)
axes[2].set_title("Image + Mask")
axes[2].axis("off")

plt.tight_layout()
plt.show()
